# Stage 0. Data Harmonization and EDA

Notebook ini membangun manifest situs palm dari metadata_en.csv dan video telapak tangan di folder palmas, menurunkan label anemik memakai ambang dewasa sesuai gender, lalu menampilkan eksplorasi distribusi hemoglobin, severity, dan demografi. Berbeda dari konjungtiva yang menggabungkan dua dataset, palm hanya satu populasi (Valles-Coral dkk. 2025, UNSM Tarapoto, Peru) sehingga struktur EDA lebih sederhana namun tetap patuh pada kontrak manifest generik proyek.

## Environment Setup

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent))

import pandas as pd
import matplotlib.pyplot as plt

from configs import paths
from src.common import manifest as manifest_utils
from src.sites.palm import data

print(paths.dataset_root("palm"))

## Build Manifest

Setiap baris manifest mewakili satu partisipan dengan satu video palm. Severity diturunkan dari kolom one-hot Normal/Mild/Moderate lalu dipetakan ke kosakata severity bersama proyek (Non-Anemic/Mild/Moderate/Severe), sedangkan label anemik diturunkan dari ambang hemoglobin dewasa sesuai gender, konsisten dengan pendekatan situs konjungtiva. Baris tanpa kategori keparahan jelas atau tanpa video yang berhasil diunduh dilewati otomatis.

In [ ]:
manifest = data.build_manifest(save=False)
print("shape", manifest.shape)
manifest.head()

## Dataset Composition

Ringkasan jumlah sampel, keseimbangan label anemik, distribusi severity, dan statistik demografi. Jumlah baris lebih kecil dari 909 partisipan asli karena sebagian video (43 video) masih menunggu retry unduhan akibat kuota akses anonim Google Drive, dan baris dengan kategori keparahan ambigu ikut dilewati.

In [ ]:
data.summarize(manifest)

## Hemoglobin Distribution

Distribusi kadar hemoglobin pada populasi dewasa muda (18-25 tahun) Peru. Garis putus-putus menandai ambang anemia dewasa WHO, 13 g/dL untuk laki-laki dan 12 g/dL untuk perempuan.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(manifest.loc[manifest["gender"] == "M", "hb_gdl"].dropna(), bins=25, alpha=0.6, label="Male")
ax.hist(manifest.loc[manifest["gender"] == "F", "hb_gdl"].dropna(), bins=25, alpha=0.6, label="Female")
ax.axvline(13.0, color="steelblue", linestyle="--", label="male threshold")
ax.axvline(12.0, color="darkorange", linestyle="--", label="female threshold")
ax.set_xlabel("Hemoglobin (g/dL)")
ax.set_ylabel("Count")
ax.set_title("Hemoglobin Distribution by Gender")
ax.legend()
plt.show()

## Class Balance and Severity

Keseimbangan label anemia dan distribusi tiga kelas severity yang tersedia pada dataset ini (Non-Anemic, Mild, Moderate, tanpa kasus Severe).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

manifest["anemic"].value_counts().sort_index().plot(kind="bar", ax=axes[0])
axes[0].set_title("Anemia Label Balance")
axes[0].set_xlabel("Anemic")
axes[0].set_ylabel("Count")

severity_order = ["Non-Anemic", "Mild", "Moderate"]
manifest["severity"].value_counts().reindex(severity_order).plot(kind="bar", ax=axes[1], color="salmon")
axes[1].set_title("Severity Distribution")
axes[1].set_xlabel("Severity")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

## Stratified Patient Split

Membagi data menjadi train, validation, dan test secara terstratifikasi per label anemik. Setiap baris mewakili satu pasien sehingga pembagian ini setara dengan pembagian berbasis pasien, sama seperti situs konjungtiva.

In [ ]:
manifest = manifest_utils.assign_stratified_split(manifest)
print(manifest.groupby(["dataset", "split"]).size().to_string())

## Save Manifest

In [ ]:
output_path = paths.outputs_dir("palm") / "manifest.csv"
manifest.to_csv(output_path, index=False)
print("manifest saved to", output_path)